In [ ]:
import torch
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')  # Add parent directory to path
from Embedder import DNAEmbedding

# Simulate a DNA sequence: ATCG repeated, then TATA
sequence = torch.tensor([
    [1,0,0,0], # A - position 0
    [0,1,0,0], # T - position 1
    [0,0,1,0], # C - position 2
    [0,0,0,1], # G - position 3
    [1,0,0,0], # A - position 4
    [0,1,0,0], # T - position 5
    [0,0,1,0], # C - position 6
    [0,0,0,1], # G - position 7
    [0,1,0,0], # T - position 8  ← TATA starts
    [1,0,0,0], # A - position 9
    [0,1,0,0], # T - position 10
    [1,0,0,0], # A - position 11
]).float().unsqueeze(0)  # Add batch dimension: (1, 12, 4)

embedder = DNAEmbedding(vocab_size=4, d_model=4)
embedded_sequence = embedder(sequence)

print(f"Input shape: {sequence.shape}")      # (1, 12, 4)
print(f"Embedded shape: {embedded_sequence.shape}")   # (1, 12, 64)
print(f"\nT at position 1: {embedded_sequence[0, 1, :5]}")   # First 5 dims
print(f"T at position 8: {embedded_sequence[0, 8, :5]}")     # First 5 dims
print(f"Are they identical? {torch.equal(embedded_sequence[0,1], embedded_sequence[0,8])}")

In [ ]:
def create_positional_encoding(seq_len, d_model):
    """
    Create positional encodings using sine/cosine functions.
    
    Why sine/cosine? 
    - Different frequencies encode position uniquely
    - Model can learn relative positions (pos 5 vs pos 10)
    
    Formula:
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    
    Args:
        seq_len: length of sequence
        d_model: embedding dimension
    
    Returns:
        (seq_len, d_model) positional encodings
    """
    position = torch.arange(seq_len).unsqueeze(1).float()  # (seq_len, 1)
    
    # Create different frequencies for each dimension
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                        -(math.log(10000.0) / d_model))
    
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)  # Even indices
    pe[:, 1::2] = torch.cos(position * div_term)  # Odd indices
    
    return pe

In [ ]:
import matplotlib.pyplot as plt
import math

pe = create_positional_encoding(seq_len=50, d_model=64)

# Plot first 4 dimensions across positions
plt.figure(figsize=(12, 4))
plt.plot(pe[:, 0], label='dim 0 (sin, low freq)')
plt.plot(pe[:, 1], label='dim 1 (cos, low freq)')
plt.plot(pe[:, 4], label='dim 4 (sin, higher freq)')
plt.plot(pe[:, 5], label='dim 5 (cos, higher freq)')
plt.xlabel('Position')
plt.ylabel('Encoding Value')
plt.title('Positional Encoding: Different frequencies encode position')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Positional encoding shape: {pe.shape}")
print(f"\nPosition 0: {pe[0, :5]}")
print(f"Position 10: {pe[10, :5]}")
print(f"Position 30: {pe[30, :5]}")

In [ ]:
pe = create_positional_encoding(seq_len=embedded_sequence.size(1), d_model=embedded_sequence.size(2))
#Combining embedding + positional encoding
embedded_with_pos = embedded_sequence + pe.unsqueeze(0)  # Add batch dimension to PE

print(f"Embedded only: {embedded_sequence[0, 1, :5]}")
print(f"Positional encoding: {pe[1, :5]}")
print(f"Combined: {embedded_with_pos[0, 1, :5]}")

# Use sequence as Q, K, V (simplified - normally would project first)
output, attn_weights = simple_attention(embedded_with_pos, embedded_with_pos, embedded_with_pos)

output.shape, attn_weights.shape, attn_weights.sum(dim=-1)